In [1]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt


In [2]:
old_df = pd.read_parquet("../data/processed/merged.parquet")

In [8]:
df = pd.read_parquet("../data/processed/processed.parquet")

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   parent_asin                    20000 non-null  str    
 1   product_title                  20000 non-null  str    
 2   features                       20000 non-null  object 
 3   description                    20000 non-null  object 
 4   categories                     20000 non-null  object 
 5   details                        20000 non-null  object 
 6   price                          11212 non-null  float64
 7   derived_avg_rating             8669 non-null   float64
 8   max_helpful_vote               8669 non-null   float64
 9   n_reviews                      20000 non-null  int64  
 10  review_text                    8669 non-null   str    
 11  candidate_review_title         8669 non-null   str    
 12  candidate_review_text          8669 non-null   str    
 1

In [10]:
df.head()

,parent_asin,product_title,features,description,categories,details,price,derived_avg_rating,max_helpful_vote,n_reviews,review_text,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
0,B0BBYGZMTT,"Dishwasher Magnet Clean Dirty Sign, Universal ...",[🌟APPLY TO ANY DISHWASHER! — Our dishwasher ma...,"[dish washer magnet clean, dishwasher sign cle...","[Appliances, Parts & Accessories, Dishwasher P...","[(Package Dimensions, ""7.05 x 2.13 x 0.55 inch...",7.99,4.924324,8.0,185,Perfect for our home: This takes the guessing ...,This will make your kitchen run smoother and e...,I love this on my dishwasher. There are adults...,8.0
1,B07F4K3HXJ,Finum Reusable Stainless Steel Coffee and Tea ...,[Permanent filter that is suitable for brewing...,[Finum Brewing Basket Given the delicate tissu...,"[Small Appliance Parts & Accessories, Coffee &...","[(Material, ""Stainless Steel""), (Color, ""Black...",16.50,4.706827,204.0,249,I really like these brewing filters for loose ...,Inexpensive and perfect for the job,This little lightweight infuser is one of the ...,204.0
2,B0C5F6QZVH,"Pureline EDR5RXD1, 4396508 Replacement for Whi...",[Triple Action Filtration: Pureline filters us...,[Pureline manufactures generic refrigerator wa...,"[Appliances, Parts & Accessories, Refrigerator...","[(Package Dimensions, ""10.5 x 2.5 x 2.5 inches...",19.99,4.909091,3.0,44,"Great deal: Fast delivery, great price! Reliab...",Good Replacement for Whirlpool 4396508 Water F...,I purchased the water filter to replace a Whir...,3.0
3,B00DM8K0FM,General Electric WD12X10136 Dishrack Roller,"[Manufacturer model # WD12X10136, Genuine Repl...","[Product Description, This is a genuine replac...","[Appliances, Parts & Accessories, Dishwasher P...","[(Manufacturer, ""General Electric""), (Part Num...",12.99,4.000000,0.0,3,Perfect. Should have been shipped with the uni...,"Fits Perfectly, Installs Easily",These fit perfectly as replacement parts in my...,0.0
4,B00B3WEOI8,"Brew Rite Coffee Filter, 3"" and 3 1/2"" Disc, W...",[Keep food fresher longer by wrapping with the...,[Brew Rite Disc Coffee Filter 100 Ct.],"[Small Appliance Parts & Accessories, Coffee &...","[(Product Dimensions, ""0.5 x 4.5 x 6.5 inches""...",4.61,4.222222,1.0,9,"ok: These fit, but I wanted the ones that fit ...",Super thin paper,This product is a VERY THIN paper product. Ea...,1.0


In [11]:
missingness = pd.DataFrame(
    {"n_missing": df.isna().sum(), "prop_missing": df.isna().mean()}
).sort_values("prop_missing", ascending=False)

missingness

,n_missing,prop_missing
derived_avg_rating,11331,0.56655
max_helpful_vote,11331,0.56655
review_text,11331,0.56655
candidate_review_title,11331,0.56655
candidate_review_text,11331,0.56655
candidate_review_helpful_vote,11331,0.56655
price,8788,0.43940
parent_asin,0,0.00000
product_title,0,0.00000
features,0,0.00000


In [12]:
df.describe()

,price,derived_avg_rating,max_helpful_vote,n_reviews,candidate_review_helpful_vote
count,11212.000000,8669.000000,8669.000000,20000.00000,8669.000000
mean,88.479685,4.256767,5.083977,3.53030,5.083977
std,310.481798,1.092489,38.984341,23.69879,38.984341
min,0.010000,1.000000,0.000000,0.00000,0.000000
25%,14.990000,4.000000,0.000000,0.00000,0.000000
50%,26.990000,4.750000,0.000000,0.00000,0.000000
75%,59.950000,5.000000,2.000000,1.00000,2.000000
max,7691.010000,5.000000,1835.000000,1220.00000,1835.000000
